# <span style="color:teal">2.3: AI Exploration: Choose Your Own Adventure Software</span>

This notebook contains a python framework for creating Choose Your Own Adventure style stories.
The goals for this notebook are to:

   -  **Use the software to come up with your own story** 
   -  **Understand Code generated from another source**
   -  **Be creative & problem solving focused to come up with improvements for the software**
   -  **Implement ideas using modern AI**

## <span style="color:Blue">*Section 1: Understanding The Framework*</span>
- Begin by running the 3 cells below to see how the software functions from a user perspective.
- Create your own story! I have starter choices and story in there, but feel free to start completely over!
- If you see something you want to modify, give it a try! You can always CTRL-Z

In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import json
import os

# Define the filename for your save data
save_file = "my_story_data.json"

# Try to load existing story data, otherwise use the default
if os.path.exists(save_file):
    with open(save_file, "r") as f:
        story_data = json.load(f)
    print(f"Story loaded from {save_file}! Found {len(story_data)} scenes.")
else:
    # Default initialization
    story_data = {
        "start": {
            "text": "You walk into school on the first day",
            "choices": { 
                "Mog the Freshmen": "mog_freshmen",
                "Be kind to the Freshman": "kind_to_freshmen"
            }
        }
    }
    print("New story engine initialized! Run the next cells to build and play.")

New story engine initialized! Run the next cells to build and play.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Create a dedicated container for your choices
choices_container = widgets.VBox()
choices = []

# 2. Refactor refresh_ui to ONLY update the container's children
def refresh_ui(choices):
    rows = [widgets.HBox([c_text, c_dest]) for c_text, c_dest in choices]
    choices_container.children = rows

def add_choice(b=None):
    num_choices = len(choices) + 1
    c_text = widgets.Text(description=f"Choice {num_choices} ", placeholder="Text the player sees")
    c_dest = widgets.Text(description=f"Dest {num_choices}", placeholder="Node ID this leads to")
    choices.append((c_text, c_dest))
    refresh_ui(choices)
    
def clear_choices():
    choices.clear()
    refresh_ui(choices)

def save_node(b):
    new_choices = {}
    for c_text, c_dest in choices:
        if c_text.value and c_dest.value:
            new_choices[c_text.value] = c_dest.value
    
    # Save to our global dictionary[cite: 1]
    story_data[node_id_input.value] = {
        "text": story_text_input.value,
        "choices": new_choices
    }
    
    # Send save text to our dedicated Output widget[cite: 1]
    with creator_out:
        clear_output()
        print(f"✅ Scene '{node_id_input.value}' saved successfully!")
        print(f"Current scenes in memory: {list(story_data.keys())}")
        
        # Clear inputs for the next scene[cite: 1]
        node_id_input.value = ""
        story_text_input.value = ""
        clear_choices()

def save_to_disk(b):
    with open("my_story_data.json", "w") as f:
        json.dump(story_data, f, indent=4)
    with creator_out:
        clear_output()
        print(f"💾 Story permanently saved to disk! ({len(story_data)} scenes)")

# 3. Create UI Widgets for the editor[cite: 1]
creator_out = widgets.Output()

html_title = widgets.HTML("<h2>📜 Story Creator</h2><p>Create or overwrite a scene (node). Use the exact <b>Node ID</b> you used as a destination in previous choices.</p>")
node_id_input = widgets.Text(description="Node ID:", placeholder="e.g., clear_path")
story_text_input = widgets.Textarea(description="Story Text:", layout=widgets.Layout(width='80%', height='100px'))

save_btn = widgets.Button(description="Save Scene", button_style='success', icon='save')
add_btn = widgets.Button(description="Add Story Branch", button_style='success', icon='plus') 

# NEW: Create the disk save button
disk_save_btn = widgets.Button(description="Export to File", button_style='primary', icon='download')

# 4. Build the main editor UI using the dynamic choices_container[cite: 1]
editor_ui = widgets.VBox([
    html_title,
    node_id_input,
    story_text_input,
    widgets.HTML("<b>Add Choices (Leave blank for an ending scene):</b>"),
    choices_container,  
    add_btn,
    widgets.HBox([save_btn, disk_save_btn]), # Put both save buttons next to each other
    creator_out
])

# 5. Connect buttons and display[cite: 1]
add_btn.on_click(add_choice)
save_btn.on_click(save_node)
disk_save_btn.on_click(save_to_disk) # Connect the new button

add_choice() 
display(editor_ui)

In [ ]:
def save_to_disk(b):
    with open("my_story_data.json", "w") as f:
        json.dump(story_data, f, indent=4)
    with creator_out:
        clear_output()
        print(f"💾 Story permanently saved to disk! ({len(story_data)} scenes)")

# 3. Create UI Widgets for the editor[cite: 1]
creator_out = widgets.Output()

html_title = widgets.HTML("<h2>📜 Story Creator</h2><p>Create or overwrite a scene (node). Use the exact <b>Node ID</b> you used as a destination in previous choices.</p>")
node_id_input = widgets.Text(description="Node ID:", placeholder="e.g., clear_path")
story_text_input = widgets.Textarea(description="Story Text:", layout=widgets.Layout(width='80%', height='100px'))

save_btn = widgets.Button(description="Save Scene", button_style='success', icon='save')
add_btn = widgets.Button(description="Add Story Branch", button_style='success', icon='plus') 

# NEW: Create the disk save button
disk_save_btn = widgets.Button(description="Export to File", button_style='primary', icon='download')

# 4. Build the main editor UI using the dynamic choices_container[cite: 1]
editor_ui = widgets.VBox([
    html_title,
    node_id_input,
    story_text_input,
    widgets.HTML("<b>Add Choices (Leave blank for an ending scene):</b>"),
    choices_container,  
    add_btn,
    widgets.HBox([save_btn, disk_save_btn]), # Put both save buttons next to each other
    creator_out
])

# 5. Connect buttons and display[cite: 1]
add_btn.on_click(add_choice)
save_btn.on_click(save_node)
disk_save_btn.on_click(save_to_disk) # Connect the new button

add_choice() 
display(editor_ui)

In [ ]:
player_out = widgets.Output()

def play_scene(node_id):
    with player_out:
        clear_output()
        
        # Check if the player wandered off the map
        if node_id not in story_data:
            display(widgets.HTML(f"<h3 style='color:red;'>Error 404: Scene Not Found</h3>"))
            print(f"The node '{node_id}' hasn't been created yet.")
            print("Go back to the Story Creator to write it!")
            return
            
        node = story_data[node_id]
        
        # Display the story text
        display(widgets.HTML(f"<p style='font-size:16px; line-height:1.5;'>{node['text']}</p>"))
        display(widgets.HTML("<hr>"))
        
        # If there are no choices, it's an ending
        if not node.get("choices"):
            display(widgets.HTML("<b>-- THE END --</b>"))
            restart_btn = widgets.Button(description="Play Again", button_style='info', icon='refresh')
            restart_btn.on_click(lambda b: play_scene("start"))
            display(restart_btn)
            return
            
        # Generate buttons for choices
        buttons = []
        for choice_text, next_node in node["choices"].items():
            btn = widgets.Button(description=choice_text, layout=widgets.Layout(width='auto', min_width='200px'))
            
            # Use a closure to capture the correct next_node for each button
            def make_callback(destination):
                return lambda b: play_scene(destination)
                
            btn.on_click(make_callback(next_node))
            buttons.append(btn)
            
        display(widgets.VBox(buttons))

# Start the game
display(player_out)
play_scene("start")

## <span style="color:Magenta">*The Assignment: Design Thinking & Implementation*</span>
- This framework was originally generated using Google Gemini. I then customized it to better fit my needs
- One area I worked on in particular was expanding beyond two options and adding the button to do so.
- This process was mostly me looking at how it did one portion, then using my knowledge to create more.
- When I was stuck, I asked gemini for help on particulars. It was very helpful in debugging!
  
- I'd like you try and do the same. Below, create a markdown cell ('b' to create a new cell, click on the left side of it and press 'm' to change to markdown)
- In this new cell, communicate in human language what you would like to change or add about the functionality of the code.
- After that, upload this file to an AI tool of your choosing and communicate the change you would like to make to it.
- See how close it gets to your idea, and keep iterating till you have what you want!